# Spark Structured Streaming


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
from config import TOPIC_FATIGA


spark = (
    SparkSession.builder
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1')  # Conector Kafka
    .master('local[*]')
    .appName('SportsSoccerTelemetry')
    .getOrCreate()
)


In [2]:
# TODO: Define schema for Kafka messages
schema = StructType([
    StructField("schema_version", StringType(), True),
    StructField("player_id", StringType(), True),
    StructField("event_id", StringType(), True),
    StructField("match_id", IntegerType(), True),
    StructField("session_id", StringType(), True),
    StructField("timestamp", DoubleType(), True), 
    StructField("bpm", IntegerType(), True),
    StructField("pos_x", DoubleType(), True),
    StructField("pos_y", DoubleType(), True),
    StructField("speed_kmh", DoubleType(), True),
    StructField("acceleration_ms2", DoubleType(), True)
])

In [3]:
# TODO: Define the topic name for Kafka producer
KAFKA_TOPIC = TOPIC_FATIGA

In [4]:
df = (
    spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', "localhost:9092")
    .option('subscribe', KAFKA_TOPIC)                     
    .option('startingOffsets', 'earliest')                
    .load()
)

In [5]:
# TODO: Parse message with defined schema
events = (
    df.selectExpr("CAST(value AS STRING) AS json_str")
      .select(F.from_json(F.col("json_str"), schema).alias("data"))
      .select("data.*")
)
events_with_time = events.withColumn(
    "event_time",
    F.to_timestamp(F.col("timestamp") / 1000)
)

In [6]:
# TODO: Define aggregations by player within the window
aggregated_df = (
    events_with_time
    .withWatermark("event_time", "2 minutes") 
    .groupBy(
        F.window("event_time", "1 minute", "30 seconds"), 
        "match_id", 
        "player_id"
    )
    .agg(
        F.count("*").alias("numero_eventos"),
        F.round(F.avg("bpm"), 2).alias("fc_media"),
        F.round(F.stddev("bpm"), 2).alias("fc_std_dev"),
        F.max("bpm").alias("fc_maxima")
    )
)


In [7]:
# TODO: Write the output to console for debugging purposes
query = (
    aggregated_df.writeStream
    .outputMode("update") 
    .format("console") 
    .option("truncate", "false") 
    .start()
)

query.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/conda/lib/python3.13/socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 